# DFN-Based Optimization of NFPP Sodium-Ion Cells within an Integrated Plant–Network Digital Twin Framework for Solar–BESS Microgrids

This notebook implements the complete research pipeline for the DFN-based optimization and the multi-feeder network state realization and anomaly detection framework.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass

# Environment Setup
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        get_ipython().system('git clone https://github.com/mhizterpaul/sodium-ion-ess.git')
        get_ipython().run_line_magic('cd', 'sodium-ion-ess')
    sys.path.append(os.getcwd())

# MP API Key configuration
if 'MP_API_KEY' not in os.environ:
    os.environ['MP_API_KEY'] = getpass("Enter Materials Project API Key: ")

!pip install pybamm numpy scipy matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl
!add-apt-repository -y ppa:fenics-packages/fenics
!apt update
!apt install -y fenicsx
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

print("\n--- PERFORMANCE COMPARISON (OPTIMIZED CANDIDATE VS. NOMINAL) ---")
opt_p = optimized_res.get("metrics", {})

metrics_to_compare = [
    ("Energy [Wh]", "energy"),
    ("Power [W]", "power"),
    ("Stability Metric", "stability_metric"),
    ("Max Strain", "max_strain")
]

print(f"{'':40s} | {'Candidate Value':20s}")
print("-" * 65)
for label, key in metrics_to_compare:
    o_val = opt_p.get(key, 0.0)
    print(f"{label:40s} | {o_val:20.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

# Map optimized design vector to parameter dict
design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

# Persist validation artifact for Stage 3.1 and Stage 4
import json
with open("final_validation.json", "w") as f:
    json.dump({"optimization": optimized_res, "validation": results}, f, indent=2)

print("\nStage 3.1: Running Parameter Extraction for Simscape...")
from src.simulation.tests import StabilityValidator
stab_validator = StabilityValidator()
envelope_res = stab_validator.validate_optimized_design()
stab_validator.export_to_json(envelope_res)

print("\n--- FULL MULTIPHYSICS SIMULATION RESULTS (BESS SCENARIOS) ---")
for k, v in envelope_res.items():
    if k in ["merged_params", "ssc_params"]: continue
    print(f"{k:40s}: {v}")

print("\nPerformance Metrics Summary:")
if results:
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## Stage 4: Network State Realization using OpenDSS (Core Contribution)
We replace the old Matlab/Simscape models with a high-fidelity OpenDSS-based digital twin framework. This framework evaluates hidden downstream distribution networks connected to the known plant feeder interface.

### Experiment 1: Hidden Bus Discovery (Complexity Characterization)
We investigate whether the complexity of the hidden downstream network (number of buses) can be inferred from the boundary measurements at the plant interface alone.

In [ ]:
import numpy as np
import opendssdirect as dss
from src.power_plant.plant import OpenDSSMicrogrid

print("Running Experiment 1: Hidden Bus Discovery...")
microgrid = OpenDSSMicrogrid()

for buses in [10, 20, 40, 80]:
    microgrid.build_base_circuit()
    microgrid.generate_random_downstream_network(num_buses=buses, topology_type="Radial")
    meas = microgrid.get_boundary_measurements()
    print(f"Buses: {buses:2d} | F1 Volt: {meas['f1_voltage']:7.2f} V | F2 Volt: {meas['f2_voltage']:7.2f} V | Phase Coupling (Delta Theta): {meas['delta_theta']:+.4f}° | Total Power: {meas['p_total']:7.2f} kW")

### Experiment 2: Connectivity Experiment (Topology Identification)
Keeping the number of buses fixed, we alter only the connections/topology (Radial vs. Multi-drop) under identical total downstream demand. We test if structural differences are observable from the boundary.

In [ ]:
print("Running Experiment 2: Connectivity / Topology Experiment...")
for topo in ["Radial", "Multi-drop"]:
    microgrid.build_base_circuit()
    microgrid.generate_random_downstream_network(num_buses=20, topology_type=topo)
    meas = microgrid.get_boundary_measurements()
    print(f"Topology: {topo:10s} | F1 Volt: {meas['f1_voltage']:7.2f} V | Phase Coupling (Delta Theta): {meas['delta_theta']:+.4f}° | Power Factor Proxy: {meas['p_total']/(np.sqrt(meas['p_total']**2 + meas['q_total']**2)):.4f}")

### Experiment 3: Dynamic Load Switching
We simulate dynamic switching events (Load Connection, Motor Starting, and Feeder Disconnection) throughout the hidden network to characterize how the boundary measurements evolve under transient disturbances.

In [ ]:
print("Running Experiment 3: Dynamic Load Switching...")
for event in ["Normal", "Load Connected", "Motor Start", "Feeder Disconnected"]:
    microgrid.build_base_circuit()
    microgrid.generate_random_downstream_network(num_buses=20, topology_type="Radial")
    
    if event == "Load Connected":
        dss.Text.Command("new load.dist_load bus1=h_bus_5 kw=35.0 pf=0.95")
    elif event == "Motor Start":
        dss.Text.Command("new load.starting_motor bus1=h_bus_10 kw=50.0 pf=0.45")
    elif event == "Feeder Disconnected":
        dss.Text.Command("line.line_h_bus_15.enabled=false")
        
    meas = microgrid.get_boundary_measurements()
    print(f"Event: {event:20s} | Total Power: {meas['p_total']:7.2f} kW | Reactive Power: {meas['q_total']:7.2f} kVAR | PCC Volt: {meas['v_pcc']:7.2f} V")

### Database Construction: The Network Signature Atlas
We compile the results of these experiments to populate the Signature Map Database (Network Signature Atlas) with derived, noise-robust features instead of raw measurements.

In [ ]:
from src.power_plant.plant import run_pipeline_experiments

print("Constructing the Network Signature Atlas Database...")
atlas, builder = run_pipeline_experiments()

print(f"
Signature Map Database compiled with {len(atlas)} entries.")
print(f"{'Sig ID':8s} | {'Event Type':22s} | {'Topology':10s} | {'Phase Coupling':15s} | {'Impedance Proxy':15s}")
print("-" * 75)
for entry in atlas[:8]:
    print(f"{entry['sig_id']:8s} | {entry['event']:22s} | {entry['realization']['topology_type']:10s} | {entry['features']['feeder_phase_coupling']:+14.4f}° | {entry['features']['aggregate_impedance_proxy']:15.4f}")

### Latent State Estimation Results
We evaluate the state estimation algorithm  = \Phi(M)$ on a test boundary measurement vector to infer hidden network properties (complexity, aggregate loading, and topology).

In [ ]:
# Select a query measurement vector from the Atlas (e.g., 40 buses radial entry)
query_meas = builder.atlas[2]["boundary"] 
estimated_state = builder.estimate_state(query_meas)

print("--- LATENT STATE ESTIMATION RESULTS ---")
print("Boundary Measurements (Inputs):")
print(f"  F1 Voltage: {query_meas['f1_voltage']:.2f} V | F2 Voltage: {query_meas['f2_voltage']:.2f} V")
print(f"  Phase Angle Coupling (Delta Theta): {query_meas['delta_theta']:.4f}°")
print(f"  Active/Reactive Power Balance: {query_meas['p_total']:.2f} kW / {query_meas['q_total']:.2f} kVAR")

print("
Inferred Latent State Vector X_R = Phi(M):")
print(f"  Estimated Downstream Bus Complexity : {estimated_state['estimated_buses']} buses")
print(f"  Estimated Topology Type             : {estimated_state['estimated_topology']}")
print(f"  Estimated Effective Distance        : {estimated_state['effective_electrical_distance']:.2f} km")
print(f"  Estimated Aggregate Loading Factor  : {estimated_state['aggregate_loading_factor']:.4f}")
print(f"  Matching Atlas Signature ID         : {estimated_state['matching_sig_id']}")
print(f"  Matching Event / Operating State    : {estimated_state['matching_event']}")